# 02 · LeetCode Practice for Squarepoint
**Format for every problem:** Question → *your attempt (with tests)* → 💡 Hint → ✅ Answer → 🔑 Key takeaway

**How to use it**
1. Run the **setup cell** once (the test harness).
2. Read the question, write your solution in the code cell, and run it. The tests print ✅ / ❌.
3. Stuck for more than 10 min? Open **💡 Hint**. Stuck for more than 25 min? Open **✅ Answer**, understand it, close it, and **re-type it from memory**.
4. Read the **🔑 Takeaway** even when you solved it. It's the part to remember.
5. Put a ⭐ next to problems you want to redo on Oct 8.

**Interview protocol (say this out loud every time):**
clarify inputs & edge cases → walk through an example → state the approach + complexity → code → **test by hand** → mention follow-ups.

### Priority map

| Section | Why | Must-do | If time |
|---|---|---|---|
| **A · Stacks & expression parsing** | Glassdoor: calculator + "string → formula" | A1, A2, **A3**, A4, **A6** | A5, A7, A8 |
| **B · Nested data / JSON / regex** | Glassdoor: nested dict grouping, JSON cleaning, regex | **B1, B2, B3, B5** | B4, B6 |
| **C · Core patterns** | Standard SWE-intern mediums | C1, C2, C4, C5, **C6** | C3, C7 |
| **D · DP** | Your worry, and a trading firm likes the stock problems | D1, D2, D3, D4, **D5**, D6 | D7–D13 |

### Suggested order (fits the 10-day plan)

| Day | Problems |
|---|---|
| Sep 30 / Oct 1 | Notebook 01 Parts 1–6, then A1, A2, A3, C1, C2 |
| Oct 2 | D1, D2, D3 (DP framework) |
| Oct 3 | A4, A6, B1, B2 |
| Oct 4 | B3, B5, C6, D4, D5 |
| Oct 5 | D6, D7, D8, C4, C5 |
| Oct 6 | 1 easy redo (demo day) |
| Oct 7 | Mock sets 1 and 2 (bottom of notebook), timed |
| Oct 8 | Redo every ⭐, plus A5 / D9–D13 if fresh |

## Setup: run this first

In [ ]:
import copy, math

def _approx(x):
    """Round floats recursively so 187.35000000001 == 187.35."""
    if isinstance(x, float):
        return round(x, 6)
    if isinstance(x, dict):
        return {k: _approx(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(_approx(v) for v in x)
    return x

def check(fn, cases, norm=None):
    """cases = [(args_tuple, expected), ...]. norm = optional function applied to both sides."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            print(f"  ✗ case {i}: raised {type(e).__name__}: {e}")
            continue
        g, e_ = _approx(got), _approx(expected)
        if norm:
            try:
                g, e_ = norm(g), norm(e_)
            except Exception:
                pass
        if g == e_:
            passed += 1
        else:
            print(f"  ✗ case {i}: args={args!r}\n      expected={expected!r}\n      got     ={got!r}")
    print(f"{'✅' if passed == len(cases) else '❌'} {passed}/{len(cases)} passed")
    return passed == len(cases)

def sort_inner(groups):          # for answers where order doesn't matter (e.g. Group Anagrams)
    return sorted(sorted(g) for g in groups)

### The DP framework (read before Section D)
1. **State**: what does `dp[i]` (or `dp[i][j]`) *mean*? Say it in one sentence, e.g. "fewest coins to make amount i".
2. **Transition**: how is `dp[i]` built from smaller states? Usually a *choice*: take/skip, extend/restart, which coin, which move.
3. **Base cases**: the smallest states (empty prefix, amount 0, first element).
4. **Order and answer**: fill in an order that makes dependencies ready; say where the answer lives (`dp[n]`, `max(dp)`, ...).

Then (optionally) **optimise space**: if row i only needs row i−1, keep one or two rows.

| Pattern | Examples | Tell-tale sign |
|---|---|---|
| 1-D linear | D1, D2, D3, D11 | "number of ways / best value up to i" |
| State machine | D4, D5 | a small set of modes per step (holding / not holding) |
| Knapsack | D6 (unbounded), D13 (0/1) | pick items to hit a target |
| Subsequence | D7 | order matters, elements can be skipped |
| Two strings | D8, D12 | two sequences, compare prefixes |
| Segmentation | D9 | split a string into valid pieces |
| Grid | D10 | move right/down |

**Top-down vs bottom-up:** in an interview, `@cache` recursion is quickest to write correctly. Say "I could convert this to bottom-up to avoid recursion limits and reduce space" and do it if asked.

---
# A · Stacks & expression parsing (Glassdoor: "calculator stack", "string → formula")

## A1 · Valid Parentheses
`LC 20 · Easy` · [LeetCode](https://leetcode.com/problems/valid-parentheses/)

**Why it's here:** The 60-second warm-up for every stack problem. Interviewers sometimes use it as the opener.

**Problem.** Given a string `s` containing only `()[]{}`, return `True` if every bracket is closed by the same type, in the correct order.

**Examples.** `"()[]{}"` → `True` · `"(]"` → `False` · `"([)]"` → `False` · `"{[]}"` → `True`

In [ ]:
def isValid(s: str) -> bool:
    pass

# --- tests ---
check(isValid, [(("()",), True), (("()[]{}",), True), (("(]",), False), (("([)]",), False), (("{[]}",), True), (("(",), False), (("]",), False)])

<details><summary>💡 <b>Hint</b></summary>

Push opening brackets. On a closing bracket, the top of the stack **must** be its partner. At the end, the stack must be empty.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def isValid(s: str) -> bool:
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in pairs:                         # closing bracket
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
        else:
            stack.append(ch)
    return not stack
```

**Complexity:** O(n) time, O(n) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

"Most recent unmatched thing" ⇒ stack. Map *closing → opening* so the check is one dict lookup. Don't forget both edge cases: closing with an empty stack, and leftovers at the end.
</details>

## A2 · Evaluate Reverse Polish Notation
`LC 150 · Medium` · [LeetCode](https://leetcode.com/problems/evaluate-reverse-polish-notation/)

**Why it's here:** Postfix evaluation is the back half of every calculator. If they ask you to convert a formula string, this is how you evaluate the result.

**Problem.** Evaluate an arithmetic expression given in postfix (RPN) as a list of tokens. Operators are `+ - * /`; division **truncates toward zero**.

**Examples.** `["2","1","+","3","*"]` → `9` · `["4","13","5","/","+"]` → `6`

In [ ]:
def evalRPN(tokens: list[str]) -> int:
    pass

# --- tests ---
check(evalRPN, [((["2","1","+","3","*"],), 9), ((["4","13","5","/","+"],), 6), ((["10","6","9","3","+","-11","*","/","*","17","+","5","+"],), 22), ((["7","-2","/"],), -3)])

<details><summary>💡 <b>Hint</b></summary>

Push numbers. On an operator, pop **b first, then a**, push `a op b`. Watch out: `"-11"` is a number, not an operator, and Python's `//` floors (−7//2 = −4).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def evalRPN(tokens: list[str]) -> int:
    stack = []
    for t in tokens:
        if t in {"+", "-", "*", "/"}:
            b = stack.pop()            # right operand is on top!
            a = stack.pop()
            if t == "+":   stack.append(a + b)
            elif t == "-": stack.append(a - b)
            elif t == "*": stack.append(a * b)
            else:          stack.append(int(a / b))   # truncate toward zero
        else:
            stack.append(int(t))
    return stack[0]
```

**Complexity:** O(n) time, O(n) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Operand order matters for `-` and `/`. Use `int(a / b)` for C-style truncation. Checking `t in {ops}` instead of `t.isdigit()` handles negative numbers for free.
</details>

## A3 · Basic Calculator II
`LC 227 · Medium` · [LeetCode](https://leetcode.com/problems/basic-calculator-ii/)

**Why it's here:** **Most likely coding question.** `+ - * /` with precedence, no parentheses. Learn this template until you can write it from memory.

**Problem.** Evaluate a string expression with non-negative integers, `+ - * /` and spaces. Integer division truncates toward zero. No `eval()`!

**Examples.** `"3+2*2"` → `7` · `" 3/2 "` → `1` · `" 3+5 / 2 "` → `5`

In [ ]:
def calculate(s: str) -> int:
    pass

# --- tests ---
check(calculate, [(("3+2*2",), 7), ((" 3/2 ",), 1), ((" 3+5 / 2 ",), 5), (("14-3/2",), 13), (("1-1+1",), 1), (("42",), 42), (("2*3*4-10/3",), 21)])

<details><summary>💡 <b>Hint</b></summary>

Keep `num` (the number being read) and `op` (the operator **before** it, initially `'+'`).
When you hit the next operator (or the end of the string), apply the *previous* op:
`+` → push `num`, `-` → push `-num`, `*`/`/` → pop, combine, push.
The answer is `sum(stack)`: the stack holds terms that only need adding.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def calculate(s: str) -> int:
    stack, num, op = [], 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)          # multi-digit numbers
        if (not ch.isdigit() and ch != " ") or i == len(s) - 1:
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            op, num = ch, 0
    return sum(stack)
```

**Complexity:** O(n) time, O(n) space. (O(1) space is possible by keeping only `last_term` instead of a stack. Good follow-up answer.)
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**"Apply the previous operator when you see the next one."** High-precedence ops (`* /`) resolve immediately against the stack top; low-precedence ones (`+ -`) are deferred by pushing signed terms. Remember the `i == len(s)-1` flush for the last number.
</details>

## A4 · Basic Calculator (with parentheses & unary minus)
`LC 224 · Hard` · [LeetCode](https://leetcode.com/problems/basic-calculator/)

**Why it's here:** Adds parentheses. Labelled Hard, but the solution is short and it's a frequent follow-up to A3.

**Problem.** Evaluate a string with non-negative integers, `+`, `-`, `(`, `)`, spaces. `-` may be unary (`"-(2+3)"`). No `*` or `/`.

**Examples.** `"(1+(4+5+2)-3)+(6+8)"` → `23` · `"-(2+3)"` → `-5`

In [ ]:
def calculate224(s: str) -> int:
    pass

# --- tests ---
check(calculate224, [(("1 + 1",), 2), ((" 2-1 + 2 ",), 3), (("(1+(4+5+2)-3)+(6+8)",), 23), (("-(2+3)",), -5), (("1-(     -2)",), 3), (("- (3 + (4 + 5))",), -12)])

<details><summary>💡 <b>Hint</b></summary>

Keep a running `result` and the current `sign` (+1/−1). On `(`: **save** `(result, sign)` on the stack and start fresh.
On `)`: finish the inner result, then `result = saved_result + saved_sign * inner`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def calculate224(s: str) -> int:
    result, num, sign = 0, 0, 1
    stack = []
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num, sign = 0, (1 if ch == "+" else -1)
        elif ch == "(":
            stack.append(result)
            stack.append(sign)
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            result *= stack.pop()      # sign before the "("
            result += stack.pop()      # result before the "("
    return result + sign * num
```

**Complexity:** O(n) time, O(depth) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Parentheses = "save context, start fresh, restore context". That idea transfers to every nested-structure problem (Decode String, Number of Atoms, JSON parsing).
</details>

## A5 · Basic Calculator III (full: + - * / and parentheses)
`LC 772 · Hard (premium)` · [LeetCode](https://leetcode.com/problems/basic-calculator-iii/)

**Why it's here:** The "calculator stack question" in its most complete form. If you can write this, any calculator variant is covered.

**Problem.** Evaluate an expression with non-negative integers, `+ - * /`, parentheses and spaces. Division truncates toward zero.

**Examples.** `"2*(5+5*2)/3+(6/2+8)"` → `21` · `"(2+6*3+5-(3*14/7+2)*5)+3"` → `-12`

In [ ]:
def calculate772(s: str) -> int:
    pass

# --- tests ---
check(calculate772, [(("1+1",), 2), (("6-4/2",), 4), (("2*(5+5*2)/3+(6/2+8)",), 21), (("(2+6*3+5-(3*14/7+2)*5)+3",), -12), (("-(3+4)*2",), -14), (("(0-7)/2",), -3), (("0",), 0)])

<details><summary>💡 <b>Hint</b></summary>

Take the A3 solution and make it **recursive**: when you see `(`, call `helper(i+1)`, which evaluates until the matching `)` and returns `(value, index_of_close_paren)`.
Treat that value exactly like a number you just read.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def calculate772(s: str) -> int:
    def apply(stack, op, num):
        if op == "+":   stack.append(num)
        elif op == "-": stack.append(-num)
        elif op == "*": stack.append(stack.pop() * num)
        else:           stack.append(int(stack.pop() / num))

    def helper(i):
        stack, num, op = [], 0, "+"
        while i < len(s):
            ch = s[i]
            if ch.isdigit():
                num = num * 10 + int(ch)
            elif ch == "(":
                num, i = helper(i + 1)          # i now points at the matching ")"
            if ch in "+-*/)" or i == len(s) - 1:
                apply(stack, op, num)
                if ch == ")":
                    return sum(stack), i
                op, num = ch, 0
            i += 1
        return sum(stack), i

    return helper(0)[0]
```

**Complexity:** O(n) time, O(depth) recursion + stack.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Recursion handles nesting and the A3 stack handles precedence, so together they cover a full calculator. The alternative is Shunting-yard (A6), which generalises better to more operators and variables. Know that both exist and pick one.
</details>

## A6 · Formula string → postfix → value (Shunting-yard, with variables)
`Custom · Medium`

**Why it's here:** Glassdoor: *"The code test is string converting to formulas."* The most likely reading: parse an infix formula (maybe with variable names), convert it, and evaluate it. Dijkstra's Shunting-yard is the textbook answer.

**Problem.** Implement:
1. `to_rpn(expr) -> list[str]`: convert an infix formula with numbers (int or decimal), variable names, `+ - * / ^` and parentheses into postfix tokens. `^` is right-associative and binds tighter than `* /`.
2. `evaluate(expr, env) -> float`: evaluate the formula, looking variables up in the dict `env`.

**Examples.** `to_rpn("a + b * c")` → `["a","b","c","*","+"]` · `evaluate("price * qty - fee", {"price": 10.5, "qty": 100, "fee": 2})` → `1048.0`

In [ ]:
def to_rpn(expr: str) -> list[str]:
    pass

def evaluate(expr: str, env: dict | None = None) -> float:
    pass

# --- tests ---
check(to_rpn, [(("a + b * c",), ["a","b","c","*","+"]), (("(a+b)*c",), ["a","b","+","c","*"]), (("2 ^ 3 ^ 2",), ["2","3","2","^","^"]), (("x1 - 3.5 / y",), ["x1","3.5","y","/","-"])])
check(evaluate, [(("price * qty - fee", {"price": 10.5, "qty": 100, "fee": 2}), 1048.0), (("(a+b)/2", {"a": 3, "b": 4}), 3.5), (("2^3^2", {}), 512.0), (("1 + 2 * (3 - 1) / 4", {}), 2.0)])

<details><summary>💡 <b>Hint</b></summary>

Tokenize with a regex: `r"\d+\.?\d*|[A-Za-z_]\w*|[-+*/^()]"`.
Shunting-yard: operands go straight to output. For an operator `o`, pop operators from the op-stack to output **while** the top has higher precedence (or equal precedence and `o` is left-associative). `(` is pushed; `)` pops until `(`. At the end, pop everything.
Then evaluate the postfix exactly like A2.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
import re, operator

PREC  = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}
RIGHT = {"^"}                                   # right-associative operators
OPS   = {"+": operator.add, "-": operator.sub, "*": operator.mul,
         "/": operator.truediv, "^": operator.pow}

def tokenize(expr):
    return re.findall(r"\d+\.?\d*|[A-Za-z_]\w*|[-+*/^()]", expr)

def to_rpn(expr: str) -> list[str]:
    out, ops = [], []
    for tok in tokenize(expr):
        if tok in PREC:
            while (ops and ops[-1] in PREC and
                   (PREC[ops[-1]] > PREC[tok] or
                    (PREC[ops[-1]] == PREC[tok] and tok not in RIGHT))):
                out.append(ops.pop())
            ops.append(tok)
        elif tok == "(":
            ops.append(tok)
        elif tok == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()                            # discard "("
        else:
            out.append(tok)                      # number or variable
    while ops:
        out.append(ops.pop())
    return out

def evaluate(expr: str, env: dict | None = None) -> float:
    env = env or {}
    stack = []
    for tok in to_rpn(expr):
        if tok in OPS:
            b, a = stack.pop(), stack.pop()
            stack.append(OPS[tok](a, b))
        elif tok[0].isdigit():
            stack.append(float(tok))
        else:
            stack.append(env[tok])               # KeyError = undefined variable
    return stack[0]
```

**Complexity:** O(n) time and space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Shunting-yard = **two stacks** (output + operators) and one rule: *pop while the top binds tighter*. Say out loud what you'd add next: unary minus (treat `-` after an operator or `(` as a different token `u-`), functions like `max(a,b)`, error handling for mismatched parentheses, and `Decimal` instead of float for money.
</details>

## A7 · Decode String
`LC 394 · Medium` · [LeetCode](https://leetcode.com/problems/decode-string/)

**Why it's here:** Nested structure + stack + string building. Same "save context on `[`" pattern as A4.

**Problem.** Decode `k[encoded]`, meaning `encoded` repeated `k` times. Nesting is allowed.

**Examples.** `"3[a]2[bc]"` → `"aaabcbc"` · `"3[a2[c]]"` → `"accaccacc"`

In [ ]:
def decodeString(s: str) -> str:
    pass

# --- tests ---
check(decodeString, [(("3[a]2[bc]",), "aaabcbc"), (("3[a2[c]]",), "accaccacc"), (("2[abc]3[cd]ef",), "abcabccdcdcdef"), (("10[x]",), "x" * 10)])

<details><summary>💡 <b>Hint</b></summary>

On `[`, push `(string_so_far, k)` and reset both. On `]`, pop `(prev, k)` and set `cur = prev + cur * k`. `k` can have multiple digits.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def decodeString(s: str) -> str:
    stack, cur, k = [], "", 0
    for ch in s:
        if ch.isdigit():
            k = k * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, k))
            cur, k = "", 0
        elif ch == "]":
            prev, times = stack.pop()
            cur = prev + cur * times
        else:
            cur += ch
    return cur
```

**Complexity:** O(output length) time and space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Every "bracketed nesting" problem is the same machine: a stack of *saved outer state*. Multi-digit numbers: `k = k*10 + digit`.
</details>

## A8 · Number of Atoms (stretch)
`LC 726 · Hard` · [LeetCode](https://leetcode.com/problems/number-of-atoms/)

**Why it's here:** The other possible reading of "string → formula" (a chemical formula). It's a good recursive-descent parsing workout. Only do it if A1–A7 feel solid.

**Problem.** Given a chemical formula like `"K4(ON(SO3)2)2"`, return the count of each atom as a string: atoms in sorted order, each followed by its count if > 1.

**Examples.** `"H2O"` → `"H2O"` · `"Mg(OH)2"` → `"H2MgO2"` · `"K4(ON(SO3)2)2"` → `"K4N2O14S4"`

In [ ]:
def countOfAtoms(formula: str) -> str:
    pass

# --- tests ---
check(countOfAtoms, [(("H2O",), "H2O"), (("Mg(OH)2",), "H2MgO2"), (("K4(ON(SO3)2)2",), "K4N2O14S4"), (("Be32",), "Be32")])

<details><summary>💡 <b>Hint</b></summary>

Recursive descent with a shared index `i`: `parse()` returns a `Counter` until it hits `)`. An atom name is one uppercase letter followed by lowercase letters. After `)` or an atom, read an optional number (default 1).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import Counter

def countOfAtoms(formula: str) -> str:
    n, i = len(formula), 0

    def read_num():
        nonlocal i
        j = i
        while i < n and formula[i].isdigit():
            i += 1
        return int(formula[j:i]) if i > j else 1

    def parse():
        nonlocal i
        cnt = Counter()
        while i < n and formula[i] != ")":
            if formula[i] == "(":
                i += 1
                inner = parse()
                i += 1                          # skip ")"
                mult = read_num()
                for atom, c in inner.items():
                    cnt[atom] += c * mult
            else:
                j = i + 1
                while j < n and formula[j].islower():
                    j += 1
                atom, i = formula[i:j], j
                cnt[atom] += read_num()
        return cnt

    c = parse()
    return "".join(a + (str(c[a]) if c[a] > 1 else "") for a in sorted(c))
```

**Complexity:** O(n · depth) worst case; fine in practice.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Recursive descent = one function per grammar rule, plus a shared cursor (`nonlocal i`). `Counter` makes merging counts trivial.
</details>

---
# B · Nested data, JSON cleaning, regex (Glassdoor: "iterate a complex dictionary", "clean JSON", "regex over raw data")

## B1 · Flatten a nested dict / JSON
`Custom · Medium`

**Why it's here:** The basic skill behind "iterate through a complex dictionary object": recursively walking dicts **and** lists.

**Problem.** Flatten a nested structure of dicts and lists into a single dict with dotted keys. List items use their index. Empty dicts/lists are kept as values. A top-level empty input returns `{}`.

**Examples.** `{"a": 1, "b": {"c": 2, "d": {"e": 3}}, "f": [10, {"g": 4}]}` → `{"a": 1, "b.c": 2, "b.d.e": 3, "f.0": 10, "f.1.g": 4}`

In [ ]:
def flatten(obj, sep: str = ".") -> dict:
    pass

# --- tests ---
check(flatten, [(({"a": 1, "b": {"c": 2, "d": {"e": 3}}, "f": [10, {"g": 4}]},), {"a": 1, "b.c": 2, "b.d.e": 3, "f.0": 10, "f.1.g": 4}), (({"x": {}},), {"x": {}}), (({"a": {"b": None}},), {"a.b": None}), (({},), {}), (({"a": [[1, 2], []]},), {"a.0.0": 1, "a.0.1": 2, "a.1": []})])

<details><summary>💡 <b>Hint</b></summary>

Write a recursive `walk(node, prefix)`. If `node` is a non-empty dict, recurse on each `(key, value)`; if a non-empty list, recurse on each `(index, value)`; otherwise it's a leaf: `out[prefix] = node`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def flatten(obj, sep: str = ".") -> dict:
    out = {}
    if not obj:
        return out

    def walk(node, prefix):
        if isinstance(node, dict) and node:
            items = node.items()
        elif isinstance(node, list) and node:
            items = enumerate(node)
        else:
            out[prefix] = node                 # leaf (incl. empty containers)
            return
        for k, v in items:
            walk(v, f"{prefix}{sep}{k}" if prefix else str(k))

    walk(obj, "")
    return out
```

**Complexity:** O(total number of nodes) time; recursion depth = nesting depth.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Branch on `isinstance(node, dict)` / `list` / leaf, which covers every JSON walker. Mention the follow-ups: **unflatten** (split keys and use `setdefault`), keys that contain the separator, and an iterative version with an explicit stack for very deep inputs.
</details>

## B2 · Collect & group items from a complex nested object
`Custom · Medium`

**Why it's here:** Almost exactly the Glassdoor report: *"iterate through a complex dictionary object and collect certain items and group them in a time-efficient manner."*

**Problem.** You receive a snapshot shaped like `accounts → desks → orders → fills` (see test data). Some keys may be **missing**.
For every order whose `status` is `"FILLED"` or `"PARTIAL"`, collect its fills and group by symbol. Return
`{sym: {"qty": total_qty, "vwap": volume-weighted avg price (4 dp), "first_ts": earliest fill ts}}`.
Do it in **one pass**.

**Examples.** See `SNAPSHOT` in the cell. Expected: AAPL qty 40, vwap 101.25, first_ts 1 · MSFT qty 5, vwap 50.0, first_ts 2

In [ ]:
SNAPSHOT = {
    "accounts": [
        {"id": "A1", "desks": [
            {"name": "eq", "orders": [
                {"oid": 1, "sym": "AAPL", "status": "FILLED",
                 "fills": [{"px": 100.0, "qty": 10, "ts": 3}, {"px": 101.0, "qty": 10, "ts": 5}]},
                {"oid": 2, "sym": "MSFT", "status": "CANCELLED", "fills": []},
                {"oid": 4, "sym": "MSFT", "status": "PARTIAL", "fills": [{"px": 50.0, "qty": 5, "ts": 2}]},
            ]},
            {"name": "empty-desk"},                       # no "orders" key!
        ]},
        {"id": "A2", "desks": [
            {"name": "eq", "orders": [
                {"oid": 3, "sym": "AAPL", "status": "FILLED", "fills": [{"px": 102.0, "qty": 20, "ts": 1}]},
                {"oid": 5, "sym": "TSLA", "status": "NEW"},  # no fills, wrong status
            ]},
        ]},
        {"id": "A3"},                                     # no "desks" key!
    ]
}

def group_fills(snapshot: dict) -> dict:
    pass

# --- tests ---
check(group_fills, [((SNAPSHOT,), {"AAPL": {"qty": 40, "vwap": 101.25, "first_ts": 1}, "MSFT": {"qty": 5, "vwap": 50.0, "first_ts": 2}}), (({"accounts": []},), {})])

<details><summary>💡 <b>Hint</b></summary>

Nested loops with `.get(key, [])` so missing keys become empty lists. Accumulate into `defaultdict` running totals (`qty`, `notional = Σ px·qty`, `first_ts`) and compute VWAP once at the end.
**Don't** build a list of all fills and re-scan it per symbol: that's O(S·N).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import defaultdict

def group_fills(snapshot: dict) -> dict:
    agg = defaultdict(lambda: {"qty": 0, "notional": 0.0, "first_ts": float("inf")})
    for acct in snapshot.get("accounts", []):
        for desk in acct.get("desks", []):
            for order in desk.get("orders", []):
                if order.get("status") not in {"FILLED", "PARTIAL"}:
                    continue
                a = agg[order["sym"]]
                for f in order.get("fills", []):
                    a["qty"] += f["qty"]
                    a["notional"] += f["px"] * f["qty"]
                    a["first_ts"] = min(a["first_ts"], f["ts"])
    return {
        sym: {"qty": a["qty"], "vwap": round(a["notional"] / a["qty"], 4), "first_ts": a["first_ts"]}
        for sym, a in agg.items() if a["qty"] > 0
    }

# --- Alternative when the shape is unknown / irregular: find every dict that has "fills" anywhere
def iter_orders(node):
    if isinstance(node, dict):
        if "fills" in node and "sym" in node:
            yield node
        for v in node.values():
            yield from iter_orders(v)
    elif isinstance(node, list):
        for v in node:
            yield from iter_orders(v)
```

**Complexity:** O(total fills) time, O(#symbols) extra space. One pass.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Three things interviewers look for here: (1) **defensive access** with `.get(k, default)`, (2) **one pass with running aggregates** in a `defaultdict` (VWAP = Σpx·q / Σq, never an average of prices), (3) you can explain the generic recursive generator (`iter_orders`) if the schema is irregular. Ask clarifying questions first: *Can keys be missing? Can qty be 0? Should output be sorted?*
</details>

## B3 · Clean messy JSON records
`Custom · Medium`

**Why it's here:** Glassdoor: *"Clean JSON data with a script."* Tests practical judgement: normalising keys, type coercion, bad-data handling, dedup.

**Problem.** Given a list of raw order records, return clean records sorted by `order_id`:
1. Normalise keys to `snake_case` (`" Order ID "`, `"orderId"`, `"order_id"` → `order_id`).
2. Types: `order_id` → int, `symbol` → stripped UPPERCASE, `price` → float, `qty` → int (**missing/invalid → 0**), `side` → `"BUY"`/`"SELL"` (accept `b/buy/s/sell`, any case). Keep `ts` as a string.
3. **Drop** a record if `order_id`, `symbol`, `price` or `side` is missing or invalid.
4. **Deduplicate** by `order_id`, keeping the record with the latest `ts` (ISO-8601 strings sort chronologically).
Output keys in order: `order_id, symbol, price, qty, side, ts`.

**Examples.** See `RAW` in the cell.

In [ ]:
RAW = [
    {" Order ID ": "1001", "Symbol": " aapl ", "Price": "187.25", "Qty": "100", "Side": "buy", "ts": "2026-10-09T09:30:01"},
    {"order_id": 1002, "symbol": "MSFT", "price": 412.1, "qty": None, "side": "SELL", "ts": "2026-10-09T09:30:02"},
    {"orderId": "1001", "symbol": "AAPL", "price": "187.30", "qty": "100", "side": "B", "ts": "2026-10-09T09:30:05"},
    {"order_id": "abc", "symbol": "", "price": "N/A", "qty": "10", "side": "buy", "ts": "bad"},
    {"order_id": 1003, "symbol": "GOOG", "price": "140", "side": "hold", "ts": "2026-10-09T09:31:00"},
]
EXPECTED = [
    {"order_id": 1001, "symbol": "AAPL", "price": 187.30, "qty": 100, "side": "BUY", "ts": "2026-10-09T09:30:05"},
    {"order_id": 1002, "symbol": "MSFT", "price": 412.1, "qty": 0, "side": "SELL", "ts": "2026-10-09T09:30:02"},
]

def clean_records(raw: list[dict]) -> list[dict]:
    pass

# --- tests ---
check(clean_records, [((RAW,), EXPECTED), (([],), [])])

<details><summary>💡 <b>Hint</b></summary>

Write small helpers: `snake(key)`, `to_int(x)`, `to_float(x)` that return `None` on failure (`try/except (TypeError, ValueError)`), and a `SIDE = {"b": "BUY", "buy": "BUY", ...}` map.
For camelCase use `re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", k)`, then replace spaces/hyphens with `_` and lowercase.
Dedup with a dict `best[order_id]`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
import re

SIDE = {"b": "BUY", "buy": "BUY", "s": "SELL", "sell": "SELL"}

def snake(key: str) -> str:
    key = key.strip()
    key = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", key)   # orderId -> order_Id
    key = re.sub(r"[\s\-]+", "_", key)                  # "Order ID" -> Order_ID
    return key.lower()

def to_int(x):
    try:
        return int(str(x).strip())
    except (TypeError, ValueError):
        return None

def to_float(x):
    try:
        return float(str(x).strip())
    except (TypeError, ValueError):
        return None

def clean_records(raw: list[dict]) -> list[dict]:
    best = {}
    for rec in raw:
        r = {snake(k): v for k, v in rec.items()}
        oid = to_int(r.get("order_id"))
        sym = str(r.get("symbol") or "").strip().upper()
        px = to_float(r.get("price"))
        side = SIDE.get(str(r.get("side") or "").strip().lower())
        if oid is None or not sym or px is None or side is None:
            continue                                      # drop invalid
        qty = to_int(r.get("qty"))
        clean = {"order_id": oid, "symbol": sym, "price": px,
                 "qty": qty if qty is not None else 0, "side": side,
                 "ts": str(r.get("ts", ""))}
        if oid not in best or clean["ts"] > best[oid]["ts"]:
            best[oid] = clean
    return sorted(best.values(), key=lambda d: d["order_id"])
```

**Complexity:** O(n log n) because of the final sort; everything else is O(n · keys).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Structure beats cleverness: **tiny helpers that return `None` on failure** make the main loop read like the spec. Talk through the choices you made: drop vs. default, logging rejected rows (in real code, count and log them rather than dropping silently), why `float` is risky for prices (`Decimal`), and that the ISO timestamp string comparison only works for a consistent format and timezone.
</details>

## B4 · Flatten Nested List Iterator
`LC 341 · Medium` · [LeetCode](https://leetcode.com/problems/flatten-nested-list-iterator/)

**Why it's here:** Iterating a nested structure *lazily*. A natural place for a Python generator, and a good way to show off `yield from`.

**Problem.** Implement `NestedIterator(nested_list)` with `next()` and `hasNext()` that yields the integers of an arbitrarily nested list in order. (LeetCode uses a `NestedInteger` wrapper; here we use plain Python lists.)

**Examples.** `[[1,1],2,[1,1]]` → `1,1,2,1,1` · `[1,[4,[6]]]` → `1,4,6`

In [ ]:
class NestedIterator:
    def __init__(self, nested: list):
        pass
    def next(self) -> int:
        pass
    def hasNext(self) -> bool:
        pass

def drain(nested):                 # test helper: don't change
    it, out = NestedIterator(nested), []
    while it.hasNext():
        out.append(it.next())
    return out

# --- tests ---
check(drain, [(([[1, 1], 2, [1, 1]],), [1, 1, 2, 1, 1]), (([1, [4, [6]]],), [1, 4, 6]), (([[], [[]], 3],), [3]), (([],), [])])

<details><summary>💡 <b>Hint</b></summary>

Keep a **stack of iterators**. `hasNext()` must do the work: advance until the top iterator produces an integer (push an iterator for each sub-list, pop exhausted ones) and cache that integer. Empty sub-lists are the edge case.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
class NestedIterator:
    def __init__(self, nested: list):
        self.stack = [iter(nested)]
        self.buf, self.has = None, False

    def _advance(self):
        while self.stack:
            try:
                x = next(self.stack[-1])
            except StopIteration:
                self.stack.pop()
                continue
            if isinstance(x, list):
                self.stack.append(iter(x))
            else:
                self.buf, self.has = x, True
                return

    def hasNext(self) -> bool:
        if not self.has:
            self._advance()
        return self.has

    def next(self) -> int:
        self.hasNext()
        self.has = False
        return self.buf

# The Pythonic version (say this out loud too):
def iter_flat(x):
    for item in x:
        if isinstance(item, list):
            yield from iter_flat(item)
        else:
            yield item
```

**Complexity:** O(1) amortized per element; O(depth) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Lazy iteration = explicit stack of iterators, or a recursive generator. `hasNext()` should be **idempotent** (calling it twice mustn't skip elements), so cache the lookahead.
</details>

## B5 · Regex: parse 3 pages of raw logs
`Custom · Medium`

**Why it's here:** Glassdoor: *"asked a Python regex coding question… 3 pages of raw data to process."* Nobody memorises every regex feature, but you should be fluent in groups, `\d \w \s`, `+ * ?`, non-greedy `.*?`, and `findall` / `finditer`.

**Problem.** Parse the raw log text. Return
`{"trades": {sym: {"count", "volume", "vwap" (4 dp)}}, "errors": {code: count}}`
- Only **TRADE** lines with a valid numeric `px` and `qty` count (skip malformed ones).
- Count `ERROR` lines by `code`.
- Ignore QUOTE lines and garbage.

**Examples.** AAPL: 2 trades, volume 300, vwap 187.35 · MSFT: 1 trade (the `px=abc` line is skipped) · errors: E42×2, E7×1

In [ ]:
RAW_LOG = """
[2026-10-09 09:30:00.001] INFO  TRADE sym=AAPL px=187.25 qty=100 venue=NASDAQ
[2026-10-09 09:30:00.050] INFO  QUOTE sym=AAPL bid=187.20 ask=187.30
[2026-10-09 09:30:00.120] INFO  TRADE sym=MSFT px=412.10 qty=50 venue=ARCA
[2026-10-09 09:30:00.300] ERROR code=E42 msg="stale book for AAPL"
[2026-10-09 09:30:01.000] INFO  TRADE sym=AAPL px=187.40 qty=200 venue=BATS
garbage line without structure
[2026-10-09 09:30:01.500] INFO  TRADE sym=MSFT px=abc qty=10 venue=ARCA
[2026-10-09 09:30:02.000] ERROR code=E42 msg="timeout"
[2026-10-09 09:30:02.500] ERROR code=E7 msg="reject"
"""

def parse_log(raw: str) -> dict:
    pass

# --- tests ---
check(parse_log, [((RAW_LOG,), {"trades": {"AAPL": {"count": 2, "volume": 300, "vwap": 187.35}, "MSFT": {"count": 1, "volume": 50, "vwap": 412.1}}, "errors": {"E42": 2, "E7": 1}}), (("",), {"trades": {}, "errors": {}})])

<details><summary>💡 <b>Hint</b></summary>

Two compiled patterns with **named groups**:
`r"\bTRADE\s+sym=(?P<sym>[A-Z]+)\s+px=(?P<px>\d+(?:\.\d+)?)\s+qty=(?P<qty>\d+)"`
`r"\bERROR\s+code=(?P<code>\w+)"`
Loop over `raw.splitlines()`, `search` each line, and accumulate. Because `px` must be digits, the `px=abc` line simply won't match.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
import re
from collections import defaultdict, Counter

TRADE_RE = re.compile(r"\bTRADE\s+sym=(?P<sym>[A-Z]+)\s+px=(?P<px>\d+(?:\.\d+)?)\s+qty=(?P<qty>\d+)")
ERROR_RE = re.compile(r"\bERROR\s+code=(?P<code>\w+)")

def parse_log(raw: str) -> dict:
    agg = defaultdict(lambda: [0, 0, 0.0])          # count, volume, notional
    errors = Counter()
    for line in raw.splitlines():
        if m := TRADE_RE.search(line):               # walrus: assign + test
            px, qty = float(m["px"]), int(m["qty"])
            a = agg[m["sym"]]
            a[0] += 1; a[1] += qty; a[2] += px * qty
        elif m := ERROR_RE.search(line):
            errors[m["code"]] += 1
    trades = {s: {"count": c, "volume": v, "vwap": round(n / v, 4)}
              for s, (c, v, n) in agg.items()}
    return {"trades": trades, "errors": dict(errors)}

# Handy generic trick: pull every key=value pair out of a line
# dict(re.findall(r'(\w+)=("[^"]*"|\S+)', line))
```

**Complexity:** O(total characters).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Regex survival kit: `re.compile` + **named groups** + `search` per line (not `match`, which anchors at the start). Use `\d+(?:\.\d+)?` for decimals, `"[^"]*"` or `".*?"` for quoted strings. Validate by construction, so malformed lines just don't match. In the interview, it's fine to say *"let me test this pattern on one line first"* and print the match.
</details>

## B6 · Simplify Path
`LC 71 · Medium` · [LeetCode](https://leetcode.com/problems/simplify-path/)

**Why it's here:** Short string-parsing + stack problem, and it's also a Unix concept (`.`, `..`) that could lead into Linux trivia.

**Problem.** Convert an absolute Unix path to its canonical form.

**Examples.** `"/a/./b/../../c/"` → `"/c"` · `"/../"` → `"/"` · `"/home//foo/"` → `"/home/foo"`

In [ ]:
def simplifyPath(path: str) -> str:
    pass

# --- tests ---
check(simplifyPath, [(("/home/",), "/home"), (("/../",), "/"), (("/home//foo/",), "/home/foo"), (("/a/./b/../../c/",), "/c"), (("/.../a/../b",), "/.../b")])

<details><summary>💡 <b>Hint</b></summary>

`path.split('/')` gives you the parts (with empty strings for `//`). `..` pops if possible; `.` and `''` are skipped; anything else is pushed.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def simplifyPath(path: str) -> str:
    stack = []
    for part in path.split("/"):
        if part == "..":
            if stack:
                stack.pop()
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)
```

**Complexity:** O(n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

`split` + stack + `join`. Note that `'...'` is a valid directory name.
</details>

---
# C · Core patterns (hash map, sliding window, intervals, design, BFS)

## C1 · Two Sum
`LC 1 · Easy` · [LeetCode](https://leetcode.com/problems/two-sum/)

**Why it's here:** Hash-map warm-up. It's the pattern behind a lot of harder problems.

**Problem.** Return indices of the two numbers that add up to `target` (exactly one solution; don't reuse an element).

**Examples.** `[2,7,11,15], 9` → `[0,1]`

In [ ]:
def twoSum(nums: list[int], target: int) -> list[int]:
    pass

# --- tests ---
check(twoSum, [(([2, 7, 11, 15], 9), [0, 1]), (([3, 2, 4], 6), [1, 2]), (([3, 3], 6), [0, 1])])

<details><summary>💡 <b>Hint</b></summary>

Store `value → index` as you go. For each `x`, check whether `target - x` is already stored.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def twoSum(nums: list[int], target: int) -> list[int]:
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return []
```

**Complexity:** O(n) time, O(n) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

"Have I seen the complement?" ⇒ hash map, one pass. Insert *after* checking so an element isn't paired with itself.
</details>

## C2 · Group Anagrams
`LC 49 · Medium` · [LeetCode](https://leetcode.com/problems/group-anagrams/)

**Why it's here:** The canonical "group by a computed key" problem, the same shape as B2.

**Problem.** Group strings that are anagrams of each other (any order).

**Examples.** `["eat","tea","tan","ate","nat","bat"]` → `[["bat"],["nat","tan"],["ate","eat","tea"]]`

In [ ]:
def groupAnagrams(strs: list[str]) -> list[list[str]]:
    pass

# --- tests ---
check(groupAnagrams, [((["eat","tea","tan","ate","nat","bat"],), [["bat"],["nat","tan"],["ate","eat","tea"]]), (([""],), [[""]]), ((["a"],), [["a"]])], norm=sort_inner)

<details><summary>💡 <b>Hint</b></summary>

Key = `''.join(sorted(word))` or a 26-count tuple. Use `defaultdict(list)`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import defaultdict

def groupAnagrams(strs: list[str]) -> list[list[str]]:
    groups = defaultdict(list)
    for w in strs:
        groups["".join(sorted(w))].append(w)      # or: tuple of 26 letter counts -> O(k)
    return list(groups.values())
```

**Complexity:** O(n · k log k) with sorting, O(n · k) with count tuples.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Grouping = `defaultdict(list)` + a **canonical key**. Keys must be hashable, so use a tuple, not a list.
</details>

## C3 · Top K Frequent Elements
`LC 347 · Medium` · [LeetCode](https://leetcode.com/problems/top-k-frequent-elements/)

**Why it's here:** Counter + heap, or bucket sort. Also tests whether you know `heapq` is a min-heap.

**Problem.** Return the `k` most frequent elements (any order).

**Examples.** `[1,1,1,2,2,3], k=2` → `[1,2]`

In [ ]:
def topKFrequent(nums: list[int], k: int) -> list[int]:
    pass

# --- tests ---
check(topKFrequent, [(([1, 1, 1, 2, 2, 3], 2), [1, 2]), (([1], 1), [1]), (([4, 4, 5, 5, 5, 6], 2), [4, 5])], norm=sorted)

<details><summary>💡 <b>Hint</b></summary>

`Counter(nums).most_common(k)` is O(n log n) and acceptable, but offer the O(n) bucket sort: index = frequency.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import Counter
import heapq

def topKFrequent(nums: list[int], k: int) -> list[int]:
    cnt = Counter(nums)
    return [x for x, _ in heapq.nlargest(k, cnt.items(), key=lambda p: p[1])]   # O(n log k)

def topKFrequent_bucket(nums, k):              # O(n)
    cnt = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for x, f in cnt.items():
        buckets[f].append(x)
    out = []
    for f in range(len(buckets) - 1, 0, -1):
        for x in buckets[f]:
            out.append(x)
            if len(out) == k:
                return out
    return out
```

**Complexity:** Heap: O(n log k). Bucket: O(n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Top-k ⇒ size-k heap (O(n log k)) or bucket by frequency (O(n)). State the tradeoff.
</details>

## C4 · Longest Substring Without Repeating Characters
`LC 3 · Medium` · [LeetCode](https://leetcode.com/problems/longest-substring-without-repeating-characters/)

**Why it's here:** The sliding-window template. The same idea shows up in rolling-window statistics over market data.

**Problem.** Length of the longest substring with no repeated characters.

**Examples.** `"abcabcbb"` → `3` · `"pwwkew"` → `3`

In [ ]:
def lengthOfLongestSubstring(s: str) -> int:
    pass

# --- tests ---
check(lengthOfLongestSubstring, [(("abcabcbb",), 3), (("bbbbb",), 1), (("pwwkew",), 3), (("",), 0), (("abba",), 2)])

<details><summary>💡 <b>Hint</b></summary>

Keep `last[ch]` = last index seen. When `ch` repeats **inside** the window, jump `left` to `last[ch] + 1`. Note the `"abba"` trap: `left` must never move backwards.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def lengthOfLongestSubstring(s: str) -> int:
    last, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        if ch in last and last[ch] >= left:
            left = last[ch] + 1
        last[ch] = right
        best = max(best, right - left + 1)
    return best
```

**Complexity:** O(n) time, O(alphabet) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Sliding window: expand `right` every step, shrink `left` only when the window breaks the invariant. Guard against moving `left` backwards.
</details>

## C5 · Merge Intervals
`LC 56 · Medium` · [LeetCode](https://leetcode.com/problems/merge-intervals/)

**Why it's here:** Sort-then-sweep. Very common, and it looks like merging time ranges or trading sessions.

**Problem.** Merge all overlapping intervals.

**Examples.** `[[1,3],[2,6],[8,10],[15,18]]` → `[[1,6],[8,10],[15,18]]`

In [ ]:
def merge(intervals: list[list[int]]) -> list[list[int]]:
    pass

# --- tests ---
check(merge, [(([[1, 3], [2, 6], [8, 10], [15, 18]],), [[1, 6], [8, 10], [15, 18]]), (([[1, 4], [4, 5]],), [[1, 5]]), (([[1, 4], [0, 4]],), [[0, 4]]), (([[1, 4], [2, 3]],), [[1, 4]])])

<details><summary>💡 <b>Hint</b></summary>

Sort by start. If the current start ≤ the last merged end, extend with `max(end)`; else append.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def merge(intervals: list[list[int]]) -> list[list[int]]:
    intervals.sort(key=lambda iv: iv[0])
    out = []
    for s, e in intervals:
        if out and s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out
```

**Complexity:** O(n log n).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Sort by start, then a greedy sweep. Remember `max()` for fully contained intervals (`[1,4],[2,3]`).
</details>

## C6 · LRU Cache
`LC 146 · Medium` · [LeetCode](https://leetcode.com/problems/lru-cache/)

**Why it's here:** The most common "design a data structure" question for SWE interns. Expect *"now do it without OrderedDict"*.

**Problem.** Implement `LRUCache(capacity)` with O(1) `get(key)` (−1 if missing) and `put(key, value)` that evicts the least-recently-used key when full.

**Examples.** See the op sequence in the test.

In [ ]:
class LRUCache:
    def __init__(self, capacity: int):
        pass
    def get(self, key: int) -> int:
        pass
    def put(self, key: int, value: int) -> None:
        pass

def run_ops(ops, args, cls=None):           # test helper: don't change
    cls = cls or LRUCache
    obj, out = None, []
    for op, a in zip(ops, args):
        if op == "LRUCache":
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

# --- tests ---
check(run_ops, [((["LRUCache","put","put","get","put","get","put","get","get","get"], [[2],[1,1],[2,2],[1],[3,3],[2],[4,4],[1],[3],[4]]), [None,None,None,1,None,-1,None,-1,3,4]), ((["LRUCache","put","put","get","put","get"], [[1],[1,1],[1,5],[1],[2,2],[1]]), [None,None,None,5,None,-1])])

<details><summary>💡 <b>Hint</b></summary>

`OrderedDict` has `move_to_end(key)` and `popitem(last=False)`.
Without it: dict `key → node` + a **doubly linked list** with sentinel head/tail. Most recent at the tail, evict from the head.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity: int):
        self.cap, self.d = capacity, OrderedDict()
    def get(self, key: int) -> int:
        if key not in self.d:
            return -1
        self.d.move_to_end(key)
        return self.d[key]
    def put(self, key: int, value: int) -> None:
        self.d[key] = value
        self.d.move_to_end(key)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)          # evict LRU (front)

# ---- Without OrderedDict: hash map + doubly linked list ----
class _Node:
    __slots__ = ("k", "v", "prev", "next")
    def __init__(self, k=0, v=0):
        self.k, self.v, self.prev, self.next = k, v, None, None

class LRUCacheDLL:
    def __init__(self, capacity: int):
        self.cap, self.map = capacity, {}
        self.head, self.tail = _Node(), _Node()      # sentinels
        self.head.next, self.tail.prev = self.tail, self.head
    def _remove(self, n):
        n.prev.next, n.next.prev = n.next, n.prev
    def _append(self, n):                            # insert before tail (= most recent)
        n.prev, n.next = self.tail.prev, self.tail
        self.tail.prev.next = n
        self.tail.prev = n
    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        n = self.map[key]
        self._remove(n); self._append(n)
        return n.v
    def put(self, key: int, value: int) -> None:
        if key in self.map:
            self._remove(self.map[key])
        n = _Node(key, value)
        self.map[key] = n
        self._append(n)
        if len(self.map) > self.cap:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.k]
```

**Complexity:** O(1) per op.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Hash map for O(1) lookup + linked list for O(1) reordering. **Sentinel nodes** remove every null check. (C++ tie-in: this is `std::list` + `unordered_map<key, list::iterator>`.)
</details>

## C7 · Number of Islands
`LC 200 · Medium` · [LeetCode](https://leetcode.com/problems/number-of-islands/)

**Why it's here:** Grid BFS/DFS template. If a graph problem shows up, it's usually this shape.

**Problem.** Count connected groups of `'1'` in a grid (4-directional).

**Examples.** See tests.

In [ ]:
def numIslands(grid: list[list[str]]) -> int:
    pass

# --- tests ---
check(numIslands, [(([["1","1","0","0","0"],["1","1","0","0","0"],["0","0","1","0","0"],["0","0","0","1","1"]],), 3), (([["1","1","1"],["0","1","0"],["1","1","1"]],), 1), (([["0"]],), 0)])

<details><summary>💡 <b>Hint</b></summary>

Scan every cell; when you find an unvisited `'1'`, count++ and BFS/DFS to mark the whole island. Use a `deque` for BFS (recursion depth can hit ~1000 on big grids).
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
from collections import deque

def numIslands(grid: list[list[str]]) -> int:
    if not grid:
        return 0
    R, C = len(grid), len(grid[0])
    count = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] != "1":
                continue
            count += 1
            grid[r][c] = "0"                         # mark visited on push
            q = deque([(r, c)])
            while q:
                x, y = q.popleft()
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < R and 0 <= ny < C and grid[nx][ny] == "1":
                        grid[nx][ny] = "0"
                        q.append((nx, ny))
    return count
```

**Complexity:** O(R·C) time; O(min(R, C)) BFS queue.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Grid template: `DIRS` tuple, bounds check, mark visited **when pushing** (not popping) to avoid duplicates. Ask if you may mutate the input; if not, use a `seen` set.
</details>

---
# D · Dynamic programming

## D1 · Climbing Stairs
`LC 70 · Easy` · [LeetCode](https://leetcode.com/problems/climbing-stairs/)

**Why it's here:** The DP "hello world". Use it to practise the 4-step framework.

**Problem.** You can climb 1 or 2 steps. How many distinct ways to reach step `n`?

**Examples.** `n=3` → `3` · `n=5` → `8`

In [ ]:
def climbStairs(n: int) -> int:
    pass

# --- tests ---
check(climbStairs, [((1,), 1), ((2,), 2), ((3,), 3), ((5,), 8), ((45,), 1836311903)])

<details><summary>💡 <b>Hint</b></summary>

`ways(i) = ways(i-1) + ways(i-2)`. You only need the last two values.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def climbStairs(n: int) -> int:
    a, b = 1, 1                  # ways(0), ways(1)
    for _ in range(n - 1):
        a, b = b, a + b
    return b

from functools import cache
def climbStairs_topdown(n):
    @cache
    def f(i):
        return 1 if i <= 1 else f(i - 1) + f(i - 2)
    return f(n)
```

**Complexity:** O(n) time, O(1) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Top-down (`@cache`) is fastest to write, and bottom-up lets you optimise space. When the recurrence only looks back k steps, keep k variables.
</details>

## D2 · House Robber
`LC 198 · Medium` · [LeetCode](https://leetcode.com/problems/house-robber/)

**Why it's here:** The canonical "take it or skip it" 1-D DP.

**Problem.** Max sum of non-adjacent elements.

**Examples.** `[2,7,9,3,1]` → `12`

In [ ]:
def rob(nums: list[int]) -> int:
    pass

# --- tests ---
check(rob, [(([1, 2, 3, 1],), 4), (([2, 7, 9, 3, 1],), 12), (([5],), 5), (([2, 1, 1, 2],), 4)])

<details><summary>💡 <b>Hint</b></summary>

`dp[i] = max(dp[i-1], dp[i-2] + nums[i])`: skip house i, or rob it.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def rob(nums: list[int]) -> int:
    prev2, prev1 = 0, 0            # dp[i-2], dp[i-1]
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1
```

**Complexity:** O(n) time, O(1) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Template: *state* = best answer using the first i items; *choice* = take/skip. Follow-ups: LC 213 (circular: run twice, excluding first or last) and LC 337 (tree).
</details>

## D3 · Maximum Subarray (Kadane)
`LC 53 · Medium` · [LeetCode](https://leetcode.com/problems/maximum-subarray/)

**Why it's here:** Kadane's algorithm = the best contiguous P&L streak. Quant-flavoured, and it's asked often.

**Problem.** Largest sum of a contiguous non-empty subarray.

**Examples.** `[-2,1,-3,4,-1,2,1,-5,4]` → `6`

In [ ]:
def maxSubArray(nums: list[int]) -> int:
    pass

# --- tests ---
check(maxSubArray, [(([-2, 1, -3, 4, -1, 2, 1, -5, 4],), 6), (([1],), 1), (([5, 4, -1, 7, 8],), 23), (([-3, -1, -2],), -1)])

<details><summary>💡 <b>Hint</b></summary>

`cur = max(x, cur + x)`: either extend the running subarray or restart at x. Track the best `cur`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxSubArray(nums: list[int]) -> int:
    cur = best = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best
```

**Complexity:** O(n), O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

`dp[i]` = best subarray **ending at i**. "Ending at i" states are the key trick for subarray/substring DPs. Initialise with `nums[0]`, not 0 (the all-negative case).
</details>

## D4 · Best Time to Buy and Sell Stock
`LC 121 · Easy` · [LeetCode](https://leetcode.com/problems/best-time-to-buy-and-sell-stock/)

**Why it's here:** You're interviewing at a trading firm, so expect the stock series. Know 121 → 122 → 309 → 714.

**Problem.** One buy, then one later sell. Max profit (0 if none).

**Examples.** `[7,1,5,3,6,4]` → `5`

In [ ]:
def maxProfit(prices: list[int]) -> int:
    pass

# --- tests ---
check(maxProfit, [(([7, 1, 5, 3, 6, 4],), 5), (([7, 6, 4, 3, 1],), 0), (([2, 4, 1],), 2)])

<details><summary>💡 <b>Hint</b></summary>

Track the running minimum price; the best profit is `max(p - min_so_far)`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxProfit(prices: list[int]) -> int:
    lo, best = float("inf"), 0
    for p in prices:
        lo = min(lo, p)
        best = max(best, p - lo)
    return best
```

**Complexity:** O(n), O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Running min / running max is a one-pass DP. LC 122 (unlimited trades) = sum of all positive day-to-day moves.
</details>

## D5 · Best Time to Buy and Sell Stock with Cooldown
`LC 309 · Medium` · [LeetCode](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-cooldown/)

**Why it's here:** **State-machine DP**. It's the pattern for the whole stock family and a great one to explain on a whiteboard.

**Problem.** Unlimited transactions, but after selling you must wait one day before buying again. Max profit.

**Examples.** `[1,2,3,0,2]` → `3` (buy, sell, cooldown, buy, sell)

In [ ]:
def maxProfitCooldown(prices: list[int]) -> int:
    pass

# --- tests ---
check(maxProfitCooldown, [(([1, 2, 3, 0, 2],), 3), (([1],), 0), (([2, 1],), 0), (([1, 2, 4],), 3)])

<details><summary>💡 <b>Hint</b></summary>

Three states at the end of each day: **hold** (own a share), **sold** (just sold today), **rest** (no share, free to buy).
`hold = max(hold, rest - p)`, `sold = hold + p`, `rest = max(rest, sold_prev)`. Update from the *previous* day's values.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def maxProfitCooldown(prices: list[int]) -> int:
    hold, sold, rest = float("-inf"), 0, 0
    for p in prices:
        prev_sold = sold
        sold = hold + p                  # sell what we held
        hold = max(hold, rest - p)       # keep holding, or buy (only from rest)
        rest = max(rest, prev_sold)      # cooldown finished, or keep resting
    return max(sold, rest)
```

**Complexity:** O(n), O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Draw the state machine (hold ⇄ sold → rest → hold) and write one line per arrow. LC 714 (fee): `sold = hold + p - fee`. LC 123/188 (at most k trades): add a transaction-count dimension.
</details>

## D6 · Coin Change
`LC 322 · Medium` · [LeetCode](https://leetcode.com/problems/coin-change/)

**Why it's here:** Unbounded knapsack, minimisation form. Very commonly asked.

**Problem.** Fewest coins to make `amount` (unlimited coins of each type), or −1.

**Examples.** `[1,2,5], 11` → `3`

In [ ]:
def coinChange(coins: list[int], amount: int) -> int:
    pass

# --- tests ---
check(coinChange, [(([1, 2, 5], 11), 3), (([2], 3), -1), (([1], 0), 0), (([186, 419, 83, 408], 6249), 20)])

<details><summary>💡 <b>Hint</b></summary>

`dp[a] = min(dp[a - c] + 1 for c in coins if c <= a)`, with `dp[0] = 0` and everything else `inf`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def coinChange(coins: list[int], amount: int) -> int:
    INF = float("inf")
    dp = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] != INF else -1
```

**Complexity:** O(amount × #coins) time, O(amount) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Greedy fails (coins `[1,3,4]`, amount 6), so say so to show you checked. Counting variant (LC 518): loop **coins outer**, amounts inner, to count combinations rather than permutations.
</details>

## D7 · Longest Increasing Subsequence
`LC 300 · Medium` · [LeetCode](https://leetcode.com/problems/longest-increasing-subsequence/)

**Why it's here:** Classic O(n²) DP with an O(n log n) upgrade using `bisect`. Showing the upgrade is a strong signal.

**Problem.** Length of the longest strictly increasing subsequence.

**Examples.** `[10,9,2,5,3,7,101,18]` → `4`

In [ ]:
def lengthOfLIS(nums: list[int]) -> int:
    pass

# --- tests ---
check(lengthOfLIS, [(([10, 9, 2, 5, 3, 7, 101, 18],), 4), (([0, 1, 0, 3, 2, 3],), 4), (([7, 7, 7, 7],), 1), (([4],), 1)])

<details><summary>💡 <b>Hint</b></summary>

O(n²): `dp[i] = 1 + max(dp[j] for j < i if nums[j] < nums[i])`.
O(n log n): keep `tails[k]` = smallest tail of an increasing subsequence of length k+1; for each x, `bisect_left(tails, x)` and replace or append.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
import bisect

def lengthOfLIS(nums: list[int]) -> int:
    tails = []
    for x in nums:
        i = bisect.bisect_left(tails, x)     # bisect_left => strictly increasing
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)

def lengthOfLIS_n2(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)
```

**Complexity:** O(n log n) / O(n²).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Say the O(n²) DP first, then optimise. `tails` is *not* the LIS itself, only its length is right, which is a common follow-up question.
</details>

## D8 · Longest Common Subsequence
`LC 1143 · Medium` · [LeetCode](https://leetcode.com/problems/longest-common-subsequence/)

**Why it's here:** The 2-D string DP template (also used by diff tools, which makes a nice git tie-in).

**Problem.** Length of the longest common subsequence of two strings.

**Examples.** `"abcde","ace"` → `3`

In [ ]:
def longestCommonSubsequence(a: str, b: str) -> int:
    pass

# --- tests ---
check(longestCommonSubsequence, [(("abcde", "ace"), 3), (("abc", "abc"), 3), (("abc", "def"), 0), (("bsbininm", "jmjkbkjkv"), 1)])

<details><summary>💡 <b>Hint</b></summary>

`dp[i][j]` = LCS of `a[:i]`, `b[:j]`. If `a[i-1] == b[j-1]`: `dp[i-1][j-1] + 1`, else `max(dp[i-1][j], dp[i][j-1])`. Size `(m+1)×(n+1)` so row/col 0 is the empty-string base case.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def longestCommonSubsequence(a: str, b: str) -> int:
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]
```

**Complexity:** O(mn) time; O(min(m, n)) space with a rolling row.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Two sequences ⇒ 2-D table over prefixes, with an extra row/col for the empty prefix. Same skeleton as Edit Distance (D11).
</details>

## D9 · Word Break
`LC 139 · Medium` · [LeetCode](https://leetcode.com/problems/word-break/)

**Why it's here:** String segmentation DP; combines parsing with DP, which fits this interview well.

**Problem.** Can `s` be segmented into dictionary words (reuse allowed)?

**Examples.** `"leetcode", ["leet","code"]` → `True`

In [ ]:
def wordBreak(s: str, wordDict: list[str]) -> bool:
    pass

# --- tests ---
check(wordBreak, [(("leetcode", ["leet", "code"]), True), (("applepenapple", ["apple", "pen"]), True), (("catsandog", ["cats", "dog", "sand", "and", "cat"]), False), (("", ["a"]), True)])

<details><summary>💡 <b>Hint</b></summary>

`dp[i]` = can `s[:i]` be segmented. `dp[i] = any(dp[j] and s[j:i] in words)`. Limit j using the max word length.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def wordBreak(s: str, wordDict: list[str]) -> bool:
    words = set(wordDict)
    maxlen = max(map(len, words), default=0)
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        for j in range(max(0, i - maxlen), i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    return dp[len(s)]
```

**Complexity:** O(n · L) checks (L = max word length), each O(L) to slice and hash.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

"Prefix is solvable" boolean DP. Convert the list to a `set` first. Follow-up LC 140 (return all sentences) needs memoised DFS.
</details>

## D10 · Unique Paths
`LC 62 · Medium` · [LeetCode](https://leetcode.com/problems/unique-paths/)

**Why it's here:** Grid DP basics. Also a nice combinatorics cross-check, which you can mention.

**Problem.** Robot moves only right/down in an `m × n` grid. Count paths from the top-left to the bottom-right.

**Examples.** `3, 7` → `28`

In [ ]:
def uniquePaths(m: int, n: int) -> int:
    pass

# --- tests ---
check(uniquePaths, [((3, 7), 28), ((3, 2), 3), ((1, 1), 1), ((10, 10), 48620)])

<details><summary>💡 <b>Hint</b></summary>

`dp[r][c] = dp[r-1][c] + dp[r][c-1]`. One row is enough.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def uniquePaths(m: int, n: int) -> int:
    row = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            row[c] += row[c - 1]
    return row[-1]

# Closed form: math.comb(m + n - 2, m - 1)
```

**Complexity:** O(mn) time, O(n) space (or O(1) via `math.comb`).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Grid DP rolls down to one row. Mentioning the combinatorial closed form C(m+n−2, m−1) goes down well at a quant firm.
</details>

## D11 · Decode Ways
`LC 91 · Medium` · [LeetCode](https://leetcode.com/problems/decode-ways/)

**Why it's here:** Parsing + DP with tricky zeros. Tests carefulness.

**Problem.** `'A'→1 … 'Z'→26`. Count ways to decode a digit string.

**Examples.** `"226"` → `3` · `"06"` → `0`

In [ ]:
def numDecodings(s: str) -> int:
    pass

# --- tests ---
check(numDecodings, [(("12",), 2), (("226",), 3), (("06",), 0), (("11106",), 2), (("10",), 1), (("2101",), 1)])

<details><summary>💡 <b>Hint</b></summary>

Like Climbing Stairs, but step 1 is allowed only if `s[i-1] != '0'`, and step 2 only if `10 <= int(s[i-2:i]) <= 26`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def numDecodings(s: str) -> int:
    prev2, prev1 = 1, 1 if s and s[0] != "0" else 0     # dp[0], dp[1]
    for i in range(2, len(s) + 1):
        cur = 0
        if s[i - 1] != "0":
            cur += prev1
        if 10 <= int(s[i - 2:i]) <= 26:
            cur += prev2
        prev2, prev1 = prev1, cur
    return prev1 if s else 0
```

**Complexity:** O(n), O(1).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Conditional Fibonacci. Enumerate the zero edge cases *before* coding: `"0"`, `"06"`, `"10"`, `"100"`.
</details>

## D12 · Edit Distance
`LC 72 · Medium` · [LeetCode](https://leetcode.com/problems/edit-distance/)

**Why it's here:** The most famous 2-D DP. Worth knowing even though it's rated Medium-Hard.

**Problem.** Minimum insert/delete/replace operations to turn `a` into `b`.

**Examples.** `"horse","ros"` → `3`

In [ ]:
def minDistance(a: str, b: str) -> int:
    pass

# --- tests ---
check(minDistance, [(("horse", "ros"), 3), (("intention", "execution"), 5), (("", "a"), 1), (("abc", "abc"), 0)])

<details><summary>💡 <b>Hint</b></summary>

`dp[i][j]` for prefixes. Base: `dp[i][0] = i`, `dp[0][j] = j`.
Equal chars → `dp[i-1][j-1]`; else `1 + min(dp[i-1][j] (delete), dp[i][j-1] (insert), dp[i-1][j-1] (replace))`.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def minDistance(a: str, b: str) -> int:
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])
    return dp[m][n]
```

**Complexity:** O(mn).
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

Same skeleton as LCS; only the base cases and transition change. Name each of the three moves when you explain it.
</details>

## D13 · Partition Equal Subset Sum
`LC 416 · Medium` · [LeetCode](https://leetcode.com/problems/partition-equal-subset-sum/)

**Why it's here:** 0/1 knapsack. The reversed inner loop is the classic trick.

**Problem.** Can the array be split into two subsets with equal sums?

**Examples.** `[1,5,11,5]` → `True`

In [ ]:
def canPartition(nums: list[int]) -> bool:
    pass

# --- tests ---
check(canPartition, [(([1, 5, 11, 5],), True), (([1, 2, 3, 5],), False), (([2, 2],), True), (([1],), False)])

<details><summary>💡 <b>Hint</b></summary>

Odd total → False. Otherwise, subset-sum to `total // 2`: `dp[s] |= dp[s - x]`, iterating `s` **downwards** so each number is used once.
</details>

<details><summary>✅ <b>Answer</b></summary>

```python
def canPartition(nums: list[int]) -> bool:
    total = sum(nums)
    if total % 2:
        return False
    target = total // 2
    dp = [True] + [False] * target
    for x in nums:
        for s in range(target, x - 1, -1):      # backwards => 0/1 (each item once)
            dp[s] = dp[s] or dp[s - x]
    return dp[target]

# Pythonic bitset version: reachable sums as bits of one big int
def canPartition_bits(nums):
    total = sum(nums)
    if total % 2:
        return False
    bits = 1
    for x in nums:
        bits |= bits << x
    return bool(bits >> (total // 2) & 1)
```

**Complexity:** O(n × target) time, O(target) space.
</details>

<details><summary>🔑 <b>Key takeaway</b></summary>

**0/1 knapsack → iterate capacity backwards; unbounded (Coin Change) → forwards.** That one sentence covers a whole family of problems.
</details>

---
# Mock interview sets (Oct 7): 30 minutes each, timer on, talk out loud

| Set | Problem 1 (~12 min) | Problem 2 (~18 min) |
|---|---|---|
| 1 | A2 Evaluate RPN | A3 Basic Calculator II → follow-up: add parentheses |
| 2 | B1 Flatten dict | B2 Collect & group, then explain the generic version |
| 3 | D4 Stock I | D5 Stock with cooldown → follow-up: transaction fee |
| 4 | C6 LRU Cache (OrderedDict) | …now without OrderedDict |

**Self-review checklist after each mock**
- [ ] Asked at least 2 clarifying questions before coding
- [ ] Stated the complexity before coding
- [ ] Named variables clearly (`stack`, `num`, `op`, not `a`, `b`, `c`)
- [ ] Traced one example by hand *after* coding
- [ ] Handled empty input / single element / negative numbers / whitespace
- [ ] Mentioned one follow-up or production concern (errors, Decimal for money, streaming input)